# SBI with the library of SBI benchmark github

## Import 

In [1]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

from sbi import analysis, utils
from sbi.inference import SNPE, simulate_for_sbi
from sbi.utils.user_input_checks import (
    check_sbi_inputs,
    process_prior,
    process_simulator,
)

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

import pyntbci

seed = 0
torch.manual_seed(seed)

c:\Users\s.velut\AppData\Local\anaconda3\envs\sbi-tall\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import numpy as np
import os
import jax.numpy as jnp
from jax.experimental.ode import odeint
import numpyro
import numpyro.distributions as dist
from jax import random
import torch

from tasks.sbibm.task_mcmc import MCMCTask, get_mcmc_samples, get_predictive_sample

from functools import partial
from nse import NSE, NSELoss
from sm_utils import train_with_validation as train
from torch.func import vmap

# from zuko.nn import MLP

from tasks.sbibm import get_task
from tall_posterior_sampler import diffused_tall_posterior_score, euler_sde_sampler, tweedies_approximation
from vp_diffused_priors import get_vpdiff_gaussian_score, get_vpdiff_uniform_score

## Useful general parameter

In [3]:
PATH_EXPERIMENT = "results/sbibm/"
NUM_OBSERVATION_LIST = list(np.arange(1, 4))

N_TRAIN_LIST = [1000, 3000, 10000, 30000]  # , 50000]
MAX_N_TRAIN = 1_000
# N_OBS_LIST = [1, 4, 8, 15]
N_OBS_LIST = [1,2]
MAX_N_OBS = 10
NUM_SAMPLES = 10

COV_MODES = ["GAUSS", "JAC"]

In [4]:
def setup(
    task, all=True, train_data=False, reference_data=False, reference_posterior=False
):
    kwargs = {}
    rng_key = random.PRNGKey(1)
    kwargs = {"rng_key": rng_key}
    if all:
        train_data = True
        reference_data = True
        reference_posterior = True
    if train_data:
        data = task.generate_training_data(n_simulations=MAX_N_TRAIN)
        print("Training data:", data["x"].shape, data["theta"].shape)
    if reference_data:
        data = task.generate_reference_data(
            nb_obs=len(NUM_OBSERVATION_LIST), n_repeat=MAX_N_OBS, **kwargs
        )
        print("Reference data:", data[0].shape, data[1][1].shape)
    if reference_posterior:
        num_obs_list = range(1, len(NUM_OBSERVATION_LIST) + 1)
        n_obs_list = N_OBS_LIST
        for num_obs in num_obs_list:
            for n_obs in n_obs_list:
                data = task.generate_reference_posterior_samples(
                    num_obs=num_obs, n_obs=n_obs, n_repeat=MAX_N_OBS, num_samples=NUM_SAMPLES, **kwargs
                )
        print("Posterior samples:", data.shape)
    return

In [5]:
def run_train_sgm(
    theta_train,
    x_train,
    n_epochs,
    batch_size,
    lr,
    clf_free_guidance=False,
    save_path=PATH_EXPERIMENT,
):
    # Set Device
    device = "cpu"
    if torch.cuda.is_available():
        device = "cuda:1"

    # Prepare training data
    # normalize theta
    theta_train_norm = (theta_train - theta_train.mean(dim=0)) / theta_train.std(dim=0)
    # normalize x
    x_train_norm = (x_train - x_train.mean(dim=0)) / x_train.std(dim=0)
    # replace nan by 0 (due to std in sir for n_train = 1000)
    x_train_norm = torch.nan_to_num(x_train_norm, nan=0.0, posinf=0.0, neginf=0.0)
    # dataset for dataloader
    data_train = torch.utils.data.TensorDataset(
        theta_train_norm.to(device), x_train_norm.to(device)
    )

    # Score network
    theta_dim = theta_train.shape[-1]
    x_dim = x_train.shape[-1]
    score_network = NSE(
        theta_dim=theta_dim,
        x_dim=x_dim,
        hidden_features=[256, 256, 256],
    ).to(device)

    # Train score network
    print(
        "=============================================================================="
    )
    print(
        f"Training score network: n_train = {theta_train.shape[0]}, n_epochs = {n_epochs}."
    )
    print(
        f"============================================================================="
    )
    print()

    if theta_train.shape[0] > 10000:
        # min_nb_epochs = n_epochs * 0.8 # 4000
        min_nb_epochs = 2000
    else:
        min_nb_epochs = 100

    # Train Score Network
    avg_score_net, train_losses, val_losses, best_epoch = train(
        score_network,
        dataset=data_train,
        loss_fn=NSELoss(score_network),
        n_epochs=n_epochs,
        lr=lr,
        batch_size=batch_size,
        validation_split=0.2,
        early_stopping=True,
        min_nb_epochs=min_nb_epochs,
        classifier_free_guidance=0.2 if clf_free_guidance else 0.0,
    )
    score_network = avg_score_net.module

    # Save Score Network
    os.makedirs(
        save_path,
        exist_ok=True,
    )
    torch.save(
        score_network,
        save_path + f"score_network.pkl",
    )
    torch.save(
        {
            "train_losses": train_losses,
            "val_losses": val_losses,
            "best_epoch": best_epoch,
        },
        save_path + f"train_losses.pkl",
    )

In [6]:
def run_sample_sgm(
    num_obs,
    context,
    nsamples,
    steps,  # number of ddim steps
    score_network,
    theta_train_mean,
    theta_train_std,
    x_train_mean,
    x_train_std,
    prior,
    prior_type,
    cov_mode,
    sampler_type="ddim",
    langevin="geffner",
    clip=False,
    theta_log_space=False,
    x_log_space=False,
    clf_free_guidance=False,
    save_path=PATH_EXPERIMENT,
):
    # Set Device
    device = "cpu"
    if torch.cuda.is_available():
        device = "cuda:1"

    n_obs = context.shape[0]

    # normalize context
    if x_log_space:
        context = torch.log(context)
    context_norm = (context - x_train_mean) / x_train_std
    # replace nan by 0 (due to std in sir for n_train = 1000)
    context_norm = torch.nan_to_num(context_norm, nan=0.0, posinf=0.0, neginf=0.0)

    # normalize prior
    if prior_type == "uniform":
        low_norm = (prior.low - theta_train_mean) / theta_train_std * 2
        high_norm = (prior.high - theta_train_mean) / theta_train_std * 2
        prior_norm = torch.distributions.Uniform(
            low_norm.to(device), high_norm.to(device)
        )
        prior_score_fn_norm = get_vpdiff_uniform_score(
            low_norm.to(device), high_norm.to(device), score_network.to(device)
        )
    elif prior_type == "gaussian":
        if theta_log_space:
            loc = prior.base_dist.loc
            cov = torch.diag_embed(prior.base_dist.scale.square())
        else:
            loc = prior.loc
            cov = prior.covariance_matrix
        loc_norm = (loc - theta_train_mean) / theta_train_std
        cov_norm = (
            torch.diag(1 / theta_train_std) @ cov @ torch.diag(1 / theta_train_std)
        )
        prior_norm = torch.distributions.MultivariateNormal(
            loc_norm.to(device), cov_norm.to(device)
        )
        prior_score_fn_norm = get_vpdiff_gaussian_score(
            loc_norm.to(device), cov_norm.to(device), score_network.to(device)
        )
    else:
        raise NotImplementedError

    print("=======================================================================")
    print(
        f"Sampling from the approximate posterior for observation {num_obs}: n_obs = {n_obs}, nsamples = {nsamples}."
    )
    print(f"======================================================================")

    if langevin:
        print()
        print(f"Using LANGEVIN sampler ({langevin.upper()}), clip = {clip}.")
        print()
        save_path += f"langevin_steps_400_5/"

        ext = ""
        theta_clipping_range = (None, None)
        if clip:
            theta_clipping_range = (-3, 3)
            ext = "_clip"

        if langevin == "geffner":
            samples = score_network.annealed_langevin_geffner(
                shape=(nsamples,),
                x=context_norm.to(device),
                prior_score_fn=prior_score_fn_norm,
                clf_free_guidance=clf_free_guidance,
                steps=400,
                lsteps=5,
                tau=0.5,
                theta_clipping_range=theta_clipping_range,
                verbose=True,
            ).cpu()
        elif langevin == "tamed":
            samples = score_network.predictor_corrector(
                (nsamples,),
                x=context_norm.to(device),
                steps=400,
                prior_score_fun=prior_score_fn_norm,
                lsteps=5,
                r=0.5,
                predictor_type="id",
                verbose=True,
                theta_clipping_range=theta_clipping_range,
            ).cpu()

            save_path = save_path[:-1] + "_ours/"
        else:
            raise NotImplementedError
        samples_filename = (
            save_path + f"posterior_samples_{num_obs}_n_obs_{n_obs}{ext}_prior.pkl"
        )
    else:
        print()
        print(
            f"Using {sampler_type.upper()} sampler, cov_mode = {cov_mode}, clip = {clip}."
        )
        print()

        cov_mode_name = cov_mode
        theta_clipping_range = (None, None)
        if clip:
            theta_clipping_range = (-3, 3)
            cov_mode_name += "_clip"

        cov_est, cov_est_prior = None, None
        if cov_mode == "GAUSS":
            # estimate cov for GAUSS
            cov_est = vmap(
                lambda x: score_network.ddim(
                    shape=(nsamples,), x=x, steps=100, eta=0.5
                ),
                randomness="different",
            )(context_norm.to(device))
            cov_est = vmap(lambda x: torch.cov(x.mT))(cov_est)

            if clf_free_guidance:
                x_ = torch.zeros_like(context_norm[0][None, :])  #
                cov_est_prior = vmap(
                    lambda x: score_network.ddim(
                        shape=(nsamples,), x=x, steps=100, eta=0.5
                    ),
                    randomness="different",
                )(x_.to(device))
                cov_est_prior = vmap(lambda x: torch.cov(x.mT))(cov_est_prior)

        if sampler_type == "ddim":
            save_path += f"ddim_steps_{steps}/"

            samples = score_network.ddim(
                shape=(nsamples,),
                x=context_norm.to(device),
                eta=1
                if steps == 1000
                else 0.8
                if steps == 400
                else 0.5,  # corresponds to the equivalent time setting from section 4.1
                steps=steps,
                theta_clipping_range=theta_clipping_range,
                prior=prior_norm,
                prior_type=prior_type,
                prior_score_fn=prior_score_fn_norm,
                clf_free_guidance=clf_free_guidance,
                dist_cov_est=cov_est,
                dist_cov_est_prior=cov_est_prior,
                cov_mode=cov_mode,
                verbose=True,
            ).cpu()
        else:
            save_path += f"euler_steps_{steps}/"

            # define score function for tall posterior
            score_fn = partial(
                diffused_tall_posterior_score,
                prior=prior_norm,  # normalized prior
                prior_type=prior_type,
                prior_score_fn=prior_score_fn_norm,  # analytical prior score function
                x_obs=context_norm.to(device),  # observations
                nse=score_network,  # trained score network
                dist_cov_est=cov_est,
                cov_mode=cov_mode,
            )

            # sample from tall posterior
            (
                samples,
                _,
            ) = euler_sde_sampler(
                score_fn,
                nsamples,
                dim_theta=theta_train_mean.shape[-1],
                beta=score_network.beta,
                device=device,
                debug=False,
                theta_clipping_range=theta_clipping_range,
            )

        assert (
            torch.isnan(samples).sum() == 0
        ), f"NaN in samples: {torch.isnan(samples).sum()}"

        samples_filename = (
            save_path
            + f"posterior_samples_{num_obs}_n_obs_{n_obs}_{cov_mode_name}_prior.pkl"
        )

    # unnormalize
    samples = samples.detach().cpu()
    samples = samples * theta_train_std + theta_train_mean
    if theta_log_space:
        samples = torch.exp(samples)

    # save  results
    os.makedirs(
        save_path,
        exist_ok=True,
    )
    torch.save(samples, samples_filename)

## Model and task

In [7]:
def model(prior_params,x_obs=None,n_obs=1):
    parameter = [numpyro.sample("theta_"+str(i),dist.Uniform(low=prior_params["low"][i],high=prior_params["high"][i])) for i in range(4)]
    fs = 500  # Hertz
    pr = 60  # Hertz
    amplitude = parameter[0]  # microvolts
    width = parameter[1]  # seconds
    latency = parameter[2]  # seconds
    encoding_length = 0.3  # seconds
    n_channels = 1
    snr = parameter[3] #0.05 #
    # code = parameter[4]

    # Generate codes
    V = pyntbci.stimulus.make_gold_codes()
    V = pyntbci.stimulus.modulate(V)
    V = V.repeat(int(fs / pr), axis=1)
    n_codes, n_samples = V.shape

    # Generate structure matrices
    M = pyntbci.utilities.encoding_matrix(V[:, np.newaxis, :], int(encoding_length * fs))

    # Generate flash-VEP
    time = jnp.arange(0, encoding_length, 1/fs)
    r = amplitude * (1 - ((time - latency) / width)**2) * jnp.exp(-0.5 * ((time - latency) / width)**2)

    # Generate template responses
    # print(r.shape)
    # print(jnp.transpose(M,(1, 0,2)).shape)
    T = jnp.matmul(r,jnp.transpose(M,(1, 0,2)).reshape((-1, n_codes * n_samples))).reshape(n_codes, n_samples)
    # T /= T.std(axis=1, keepdims=True)  # normalize amplitudes

    noise=0
    sig_avg = jnp.mean(T**2,axis=1)
    sig_avg_db = 10 * jnp.log10(sig_avg)
    # Calculate noise according to [2] then convert to watts
    noise_avg_db = sig_avg_db - 1/snr
    noise_avg = 10 ** (noise_avg_db / 10)
    # Generate an sample of white noise
    mean_noise = 0
    # noise = np.random.normal(mean_noise, jnp.sqrt(jnp.mean(noise_avg)), T.shape)
    noise = random.normal(random.PRNGKey(0),T[0].shape)*jnp.sqrt(jnp.mean(noise_avg))

    # plt.plot(T[0].numpy()+noise)
    # plt.show()
    for i in range(n_obs):
        if x_obs is None:
            numpyro.sample(
                f"obs_{i}",
                dist.LogNormal(
                    jnp.log(jnp.clip(T[0]+noise, 1e-10, 10000)), scale=0.001
                ),  # noq
            )
        else:
            numpyro.sample(
                f"obs_{i}",
                dist.LogNormal(jnp.log(jnp.clip(T[0]+noise, 1e-10, 10000)), scale=0.001),
                obs=x_obs[i],
            )
    # return T+noise

In [8]:
class FlashVEP(MCMCTask):
    """ 
    Model very basic of a mseq EEG response in one channel 
    """
    def __init__(self,low,high,dim_theta=4,dim_x=(63,1008),*kwargs):
        self.low = low
        self.high = high
        self.dim_theta = dim_theta
        name = "FalshVEP"
        super().__init__(name=name, prior_params={"high":high,"low":low}, model=model)
    
    def prior(self):
        return torch.distributions.Uniform(low=torch.Tensor(self.low).float(),high=torch.Tensor(self.high).float())
    
    def _simulate_one(self, rng_key, theta, n_obs):
        # define the condition
        cond = {}
        for j in range(4):
            cond[f"theta_{j}"] = theta[j]

        x = get_predictive_sample(
            rng_key=rng_key,
            model=self.model,
            cond=cond,
            n_obs=n_obs,
        )
        return x  # shape (n_obs, dim_x=10)

    def _posterior_sampler(self, rng_key, x_star, theta_star, n_obs, num_samples):
        # define the condition
        init_values = {}
        for j in range(4):
            init_values[f"theta_{j}"] = theta_star[j]

        print("init",init_values)
        samples = get_mcmc_samples(
            rng_key=rng_key,
            model=self.model,
            init_value=init_values,
            data=x_star,
            num_samples=num_samples,
            n_obs=n_obs,
        )
        return jnp.stack([samples["theta_"+str(i)] for i in range(4)]).T

## Train the algorithm

In [23]:
task = FlashVEP(low=[0.5,0.005,0.050,0.0], high=[2.0,0.05,0.400,0.5])

setup(
            task,
            all=True,
            train_data=False,
            reference_data=False,
            reference_posterior=False,
        )
task_path = PATH_EXPERIMENT + f"FlashVEP/"

n_train = 5000
batch_size = 256
n_epochs = 5000
lr = 1e-4
clf_free_guidance = False

# Define Experiment Path
save_path = (
    task_path
    + f"n_train_{n_train}_bs_{batch_size}_n_epochs_{n_epochs}_lr_{lr}/"
)

os.makedirs(save_path, exist_ok=True)

print()
print("save_path: ", save_path)
print("CUDA available: ", torch.cuda.is_available())
print()

Generating training data for 1000 simulations and 1 observations.


100%|██████████| 1000/1000 [00:14<00:00, 67.65it/s]


Saving at data/FalshVEP/dataset_n_train_1000.pkl
Training data: torch.Size([1000, 1008]) torch.Size([1000, 4])
Generating reference data for 3 observations and 10 repetitions.
Saving at data/FalshVEP/theta_true_list.pkl

Saving at data/FalshVEP/reference_observations/x_obs_10_num_1.pkl

Saving at data/FalshVEP/reference_observations/x_obs_10_num_2.pkl

Saving at data/FalshVEP/reference_observations/x_obs_10_num_3.pkl
Reference data: torch.Size([3, 4]) torch.Size([10, 1008])
Generating reference posterior samples for num_obs = 1 and n_obs = 1
init {'theta_0': Array(1.410741, dtype=float32), 'theta_1': Array(0.02215984, dtype=float32), 'theta_2': Array(0.12105183, dtype=float32), 'theta_3': Array(0.3889195, dtype=float32)}


sample: 100%|██████████| 110/110 [11:55<00:00,  6.50s/it, 1023 steps of size 7.11e-08. acc. prob=0.80]
d:\s.velut\documents\these\dcc\sbi\diffusions-for-sbi\tasks\sbibm\task.py:156: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub 

Saving at data/FalshVEP/reference_posterior_samples/true_posterior_samples_num_1_n_obs_1.pkl
Generating reference posterior samples for num_obs = 1 and n_obs = 2
init {'theta_0': Array(1.410741, dtype=float32), 'theta_1': Array(0.02215984, dtype=float32), 'theta_2': Array(0.12105183, dtype=float32), 'theta_3': Array(0.3889195, dtype=float32)}


sample: 100%|██████████| 110/110 [11:05<00:00,  6.05s/it, 1023 steps of size 6.67e-08. acc. prob=0.80]
d:\s.velut\documents\these\dcc\sbi\diffusions-for-sbi\tasks\sbibm\task.py:156: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub 

... (5 outputs left out)

sample: 100%|██████████| 110/110 [03:46<00:00,  2.06s/it, 1023 steps of size 1.75e-08. acc. prob=0.79]
d:\s.velut\documents\these\dcc\sbi\diffusions-for-sbi\tasks\sbibm\task.py:156: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub 

Saving at data/FalshVEP/reference_posterior_samples/true_posterior_samples_num_3_n_obs_1.pkl
Generating reference posterior samples for num_obs = 3 and n_obs = 2
init {'theta_0': Array(1.558864, dtype=float32), 'theta_1': Array(0.03287025, dtype=float32), 'theta_2': Array(0.20703053, dtype=float32), 'theta_3': Array(0.10734373, dtype=float32)}


sample: 100%|██████████| 110/110 [04:06<00:00,  2.24s/it, 1023 steps of size 6.18e-08. acc. prob=0.79]

Saving at data/FalshVEP/reference_posterior_samples/true_posterior_samples_num_3_n_obs_2.pkl
Posterior samples: torch.Size([10, 4])

save_path:  results/sbibm/FlashVEP/n_train_5000_bs_256_n_epochs_5000_lr_0.0001/
CUDA available:  False



In [24]:
# get training data
dataset_train = task.get_training_data(n_simulations=MAX_N_TRAIN)
theta_train = dataset_train["theta"].float()
x_train = dataset_train["x"].float()
# extract training data for given n_train
theta_train, x_train = theta_train[:n_train], x_train[:n_train]
print("Training data:", theta_train.shape, x_train.shape)

# log space transformation
if task in ["lotka_volterra", "sir", "FlashVEP"]:
    print("Transforming data to log space.")
    # theta_train = torch.log(theta_train)
    x_train = torch.log(x_train)

# compute mean and std of training data
theta_train_mean, theta_train_std = theta_train.mean(
    dim=0
), theta_train.std(dim=0)
x_train_mean, x_train_std = x_train.mean(dim=0), x_train.std(dim=0)
means_stds_dict = {
    "theta_train_mean": theta_train_mean,
    "theta_train_std": theta_train_std,
    "x_train_mean": x_train_mean,
    "x_train_std": x_train_std,
}
torch.save(means_stds_dict, save_path + f"train_means_stds_dict.pkl")

run_fn = run_train_sgm
kwargs_run = {
    "theta_train": theta_train,
    "x_train": x_train,
    "n_epochs": n_epochs,
    "batch_size": batch_size,
    "lr": lr,
    "clf_free_guidance": clf_free_guidance,
    "save_path": save_path,
            }

run_fn(**kwargs_run)

d:\s.velut\documents\these\dcc\sbi\diffusions-for-sbi\tasks\sbibm\task.py:144: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  dataset_train = torch.load(filename)


Loading training data from data/FalshVEP/dataset_n_train_1000.pkl
Training data: torch.Size([1000, 4]) torch.Size([1000, 1008])
Training score network: n_train = 1000, n_epochs = 5000.



Training epochs:   0%|          | 0/5000 [00:00<?, ?it/s]

Training epochs: 100%|██████████| 5000/5000 [02:17<00:00, 36.45it/s, lr=0.0001, train_loss=0.0712, val_loss=0.0797]

EARLY STOPPING: best validation loss 0.052728284150362015 at epoch 4768
Did not improve for 232 epochs


Training epochs:   0%|          | 0/5000 [00:00<?, ?it/s]
torch.Size([256, 4]) torch.Size([256, 63, 1008])
in NSEEEEEEEEEEEEEEEEEEEEEEEEEEE torch.Size([256, 4]) torch.Size([256, 63, 1008]) torch.Size([256]) torch.Size([256, 4])


## Sample things

In [31]:
num_obs = 1
n_obs=2
cov_mode = "GAUSS"
sampler = "ddim"  # ddim or euler
langevin = ""  #"" or geffner or tamed

# get reference observations
x_obs_100 = task.get_reference_observation(num_obs, n_repeat=MAX_N_OBS)
context = x_obs_100[:n_obs].reshape(n_obs, -1)
print("Context:", context.shape)

# Trained Score network
score_network = torch.load(
    save_path + f"score_network.pkl",
    map_location=torch.device("cpu"),
)
score_network.net_type = "default"
score_network.tweedies_approximator = tweedies_approximation

# Mean and std of training data
means_stds_dict = torch.load(save_path + f"train_means_stds_dict.pkl")
theta_train_mean = means_stds_dict["theta_train_mean"]
theta_train_std = means_stds_dict["theta_train_std"]
x_train_mean = means_stds_dict["x_train_mean"]
x_train_std = means_stds_dict["x_train_std"]

run_fn = run_sample_sgm
kwargs_run = {
    "num_obs": num_obs,
    "context": context,
    "nsamples": NUM_SAMPLES,
    "score_network": score_network,
    "steps": 1000
    if cov_mode == "GAUSS"
    else 400,  # corresponds to the equivalent time setting from section 4.1
    "theta_train_mean": theta_train_mean,  # for (un)normalization
    "theta_train_std": theta_train_std,  # for (un)normalization
    "x_train_mean": x_train_mean,  # for (un)normalization
    "x_train_std": x_train_std,  # for (un)normalization
    "prior": task.prior(),  # for score function
    "prior_type": "uniform",
    "cov_mode": cov_mode,
    "clip": True,  # for clipping
    "sampler_type": sampler,
    "langevin": langevin,
    "theta_log_space": task in ["lotka_volterra", "sir"],
    "x_log_space": task == ["lotka_volterra","FlashVEP"],
    "clf_free_guidance": clf_free_guidance,
    "save_path": save_path,
}

run_fn(**kwargs_run)


d:\s.velut\documents\these\dcc\sbi\diffusions-for-sbi\tasks\sbibm\task.py:170: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  x_star = torch.load(filename)
C:\Users\s.velut\A

Loading reference observations from data/FalshVEP/reference_observations/x_obs_10_num_1.pkl
Context: torch.Size([2, 1008])
Sampling from the approximate posterior for observation 1: n_obs = 2, nsamples = 10.

Using DDIM sampler, cov_mode = GAUSS, clip = True.



100%|██████████| 1000/1000 [00:01<00:00, 530.27it/s]


In [37]:
theta1 = torch.load("D:/s.velut/Documents/These/DCC/SBI/results/sbibm/FlashVEP/n_train_5000_bs_256_n_epochs_5000_lr_0.0001/ddim_steps_1000/posterior_samples_1_n_obs_2_GAUSS_clip_prior.pkl")
theta2 = torch.load("D:/s.velut/Documents/These/DCC/SBI/results/sbibm/FlashVEP/n_train_5000_bs_256_n_epochs_5000_lr_0.0001/ddim_steps_1000/posterior_samples_2_n_obs_2_GAUSS_clip_prior.pkl")
theta_true = torch.load("D:/s.velut/Documents/These/DCC/SBI/data/FalshVEP/theta_true_list.pkl")
theta3 = torch.load("D:/s.velut/Documents/These/DCC/SBI/results/sbibm/FlashVEP/n_train_5000_bs_256_n_epochs_5000_lr_0.0001/ddim_steps_1000/posterior_samples_3_n_obs_2_GAUSS_clip_prior.pkl")

C:\Users\s.velut\AppData\Local\Temp\ipykernel_31112\2244210727.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  theta1 = torch.load("D:/s.velut/Documents/These/DCC/SBI/re

In [40]:
print(theta1 - theta_true[0])
print(theta2 - theta_true[1])
print(theta3 - theta_true[2])

tensor([[ 1.7009e-02,  9.8434e-04, -1.6825e-02, -6.4340e-03],
        [ 2.0132e-02,  1.0696e-03, -2.6182e-02, -8.6296e-03],
        [-8.0490e-03, -1.3893e-04, -1.0513e-02,  1.6086e-03],
        [-3.5193e-02, -3.6726e-04, -1.4254e-02,  5.7062e-03],
        [ 2.4187e-02,  1.2934e-04, -1.3558e-02,  5.9164e-03],
        [ 1.2320e-03, -1.0555e-03, -1.1180e-02, -6.1674e-03],
        [-1.4777e-03, -3.1534e-04, -2.1473e-02, -1.9931e-03],
        [-3.9270e-02, -4.8597e-04, -1.4019e-02,  1.1488e-02],
        [-2.4577e-02,  1.0279e-03, -3.6107e-02,  6.9588e-05],
        [ 7.0862e-03,  9.1794e-04, -1.9133e-02, -1.3499e-02]])
tensor([[ 0.0835,  0.0005, -0.0083, -0.0707],
        [ 0.0554, -0.0005,  0.0039, -0.0642],
        [ 0.0823, -0.0007, -0.0068, -0.0632],
        [-0.0202, -0.0002,  0.0008, -0.0588],
        [ 0.0392, -0.0007, -0.0016, -0.0514],
        [ 0.0540, -0.0002, -0.0037, -0.0538],
        [ 0.0224,  0.0017,  0.0008, -0.0670],
        [ 0.0409, -0.0003, -0.0081, -0.0699],
        [ 0

C:\Users\s.velut\AppData\Local\Temp\ipykernel_31112\3932027898.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load("D:/s.velut/Documents/These/DCC/SBI/results/sbi

{'theta_train_mean': tensor([1.2644, 0.0269, 0.2299, 0.2533]),
 'theta_train_std': tensor([0.4352, 0.0129, 0.0985, 0.1445]),
 'x_train_mean': tensor([9.9997e-11, 1.0000e-10, 8.0758e-01,  ..., 3.0075e+00, 7.8902e-01,
         6.1354e-01]),
 'x_train_std': tensor([1.0007e-13, 9.9514e-14, 6.9269e-01,  ..., 4.1917e+00, 1.9464e+00,
         1.7446e+00])}

# Test on Lokta Volkera

In [11]:
n_train = 5000
batch_size = 256
n_epochs = 5000
lr = 1e-4
clf_free_guidance = False

# Define Experiment Path
save_path = (
    task_path
    + f"n_train_{n_train}_bs_{batch_size}_n_epochs_{n_epochs}_lr_{lr}/"
)

task = get_task("lotka_volterra", save_path=save_path)
print(task)
setup(
            task,
            all=False,
            train_data=False,
            reference_data=False,
            reference_posterior=False,
        )
task_path = PATH_EXPERIMENT + f"FlashVEP/"



os.makedirs(save_path, exist_ok=True)

print()
print("save_path: ", save_path)
print()


save_path:  results/sbibm/FlashVEP/n_train_5000_bs_256_n_epochs_5000_lr_0.0001/



In [12]:
# get training data
dataset_train = task.get_training_data(n_simulations=MAX_N_TRAIN)
theta_train = dataset_train["theta"].float()
x_train = dataset_train["x"].float()
# extract training data for given n_train
theta_train, x_train = theta_train[:n_train], x_train[:n_train]
print("Training data:", theta_train.shape, x_train.shape)

# log space transformation
print("Transforming data to log space.")
theta_train = torch.log(theta_train)
x_train = torch.log(x_train)

# compute mean and std of training data
theta_train_mean, theta_train_std = theta_train.mean(
    dim=0
), theta_train.std(dim=0)
x_train_mean, x_train_std = x_train.mean(dim=0), x_train.std(dim=0)
means_stds_dict = {
    "theta_train_mean": theta_train_mean,
    "theta_train_std": theta_train_std,
    "x_train_mean": x_train_mean,
    "x_train_std": x_train_std,
}
torch.save(means_stds_dict, save_path + f"train_means_stds_dict.pkl")

run_fn = run_train_sgm
kwargs_run = {
    "theta_train": theta_train,
    "x_train": x_train,
    "n_epochs": n_epochs,
    "batch_size": batch_size,
    "lr": lr,
    "clf_free_guidance": clf_free_guidance,
    "save_path": save_path,
}

run_fn(**kwargs_run)

d:\s.velut\documents\these\dcc\sbi\diffusions-for-sbi\tasks\sbibm\task.py:144: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  dataset_train = torch.load(filename)


Loading training data from results/sbibm/FlashVEP/n_train_5000_bs_256_n_epochs_5000_lr_0.0001/lotka_volterra/dataset_n_train_1000.pkl
Training data: torch.Size([1000, 4]) torch.Size([1000, 20])
Transforming data to log space.
Training score network: n_train = 1000, n_epochs = 5000.



Training epochs:   0%|          | 4/5000 [00:00<02:22, 34.96it/s, lr=0.0001, train_loss=1.01, val_loss=0.904] 

torch.Size([256, 4]) torch.Size([256, 20])
in NSEEEEEEEEEEEEEEEEEEEEEEEEEEE torch.Size([256, 4]) torch.Size([256, 20]) torch.Size([256]) torch.Size([256, 4])
torch.Size([256, 4]) torch.Size([256, 20])
in NSEEEEEEEEEEEEEEEEEEEEEEEEEEE torch.Size([256, 4]) torch.Size([256, 20]) torch.Size([256]) torch.Size([256, 4])
torch.Size([256, 4]) torch.Size([256, 20])
in NSEEEEEEEEEEEEEEEEEEEEEEEEEEE torch.Size([256, 4]) torch.Size([256, 20]) torch.Size([256]) torch.Size([256, 4])
torch.Size([32, 4]) torch.Size([32, 20])
in NSEEEEEEEEEEEEEEEEEEEEEEEEEEE torch.Size([32, 4]) torch.Size([32, 20]) torch.Size([32]) torch.Size([32, 4])
in NSEEEEEEEEEEEEEEEEEEEEEEEEEEE torch.Size([200, 4]) torch.Size([200, 20]) torch.Size([200]) torch.Size([200, 4])
torch.Size([256, 4]) torch.Size([256, 20])
in NSEEEEEEEEEEEEEEEEEEEEEEEEEEE torch.Size([256, 4]) torch.Size([256, 20]) torch.Size([256]) torch.Size([256, 4])
torch.Size([256, 4]) torch.Size([256, 20])
in NSEEEEEEEEEEEEEEEEEEEEEEEEEEE torch.Size([256, 4]) torc

Training epochs:   0%|          | 4/5000 [00:00<02:22, 34.96it/s, lr=0.0001, train_loss=0.891, val_loss=0.879]

torch.Size([256, 4]) torch.Size([256, 20])
in NSEEEEEEEEEEEEEEEEEEEEEEEEEEE torch.Size([256, 4]) torch.Size([256, 20]) torch.Size([256]) torch.Size([256, 4])
torch.Size([256, 4]) torch.Size([256, 20])
in NSEEEEEEEEEEEEEEEEEEEEEEEEEEE torch.Size([256, 4]) torch.Size([256, 20]) torch.Size([256]) torch.Size([256, 4])
torch.Size([256, 4]) torch.Size([256, 20])
in NSEEEEEEEEEEEEEEEEEEEEEEEEEEE torch.Size([256, 4]) torch.Size([256, 20]) torch.Size([256]) torch.Size([256, 4])
torch.Size([32, 4]) torch.Size([32, 20])
in NSEEEEEEEEEEEEEEEEEEEEEEEEEEE torch.Size([32, 4]) torch.Size([32, 20]) torch.Size([32]) torch.Size([32, 4])
in NSEEEEEEEEEEEEEEEEEEEEEEEEEEE torch.Size([200, 4]) torch.Size([200, 20]) torch.Size([200]) torch.Size([200, 4])


... (785 outputs left out)

torch.Size([256, 4]) torch.Size([256, 20])
in NSEEEEEEEEEEEEEEEEEEEEEEEEEEE torch.Size([256, 4]) torch.Size([256, 20]) torch.Size([256]) torch.Size([256, 4])
torch.Size([256, 4]) torch.Size([256, 20])
in NSEEEEEEEEEEEEEEEEEEEEEEEEEEE torch.Size([256, 4]) torch.Size([256, 20]) torch.Size([256]) torch.Size([256, 4])
torch.Size([256, 4]) torch.Size([256, 20])
in NSEEEEEEEEEEEEEEEEEEEEEEEEEEE torch.Size([256, 4]) torch.Size([256, 20]) torch.Size([256]) torch.Size([256, 4])
torch.Size([32, 4]) torch.Size([32, 20])
in NSEEEEEEEEEEEEEEEEEEEEEEEEEEE torch.Size([32, 4]) torch.Size([32, 20]) torch.Size([32]) torch.Size([32, 4])
in NSEEEEEEEEEEEEEEEEEEEEEEEEEEE torch.Size([200, 4]) torch.Size([200, 20]) torch.Size([200]) torch.Size([200, 4])
torch.Size([256, 4]) torch.Size([256, 20])
in NSEEEEEEEEEEEEEEEEEEEEEEEEEEE torch.Size([256, 4]) torch.Size([256, 20]) torch.Size([256]) torch.Size([256, 4])
torch.Size([256, 4]) torch.Size([256, 20])
in NSEEEEEEEEEEEEEEEEEEEEEEEEEEE torch.Size([256, 4]) torc

Training epochs:  43%|████▎     | 2143/5000 [01:00<01:20, 35.53it/s, lr=0.0001, train_loss=0.0628, val_loss=0.0818]

torch.Size([256, 4]) torch.Size([256, 20])
in NSEEEEEEEEEEEEEEEEEEEEEEEEEEE torch.Size([256, 4]) torch.Size([256, 20]) torch.Size([256]) torch.Size([256, 4])
torch.Size([256, 4]) torch.Size([256, 20])
in NSEEEEEEEEEEEEEEEEEEEEEEEEEEE torch.Size([256, 4]) torch.Size([256, 20]) torch.Size([256]) torch.Size([256, 4])
torch.Size([256, 4]) torch.Size([256, 20])
in NSEEEEEEEEEEEEEEEEEEEEEEEEEEE torch.Size([256, 4]) torch.Size([256, 20]) torch.Size([256]) torch.Size([256, 4])
torch.Size([32, 4]) torch.Size([32, 20])
in NSEEEEEEEEEEEEEEEEEEEEEEEEEEE torch.Size([32, 4]) torch.Size([32, 20]) torch.Size([32]) torch.Size([32, 4])
in NSEEEEEEEEEEEEEEEEEEEEEEEEEEE torch.Size([200, 4]) torch.Size([200, 20]) torch.Size([200]) torch.Size([200, 4])
torch.Size([256, 4]) torch.Size([256, 20])
in NSEEEEEEEEEEEEEEEEEEEEEEEEEEE torch.Size([256, 4]) torch.Size([256, 20]) torch.Size([256]) torch.Size([256, 4])
torch.Size([256, 4]) torch.Size([256, 20])
in NSEEEEEEEEEEEEEEEEEEEEEEEEEEE torch.Size([256, 4]) torc

KeyboardInterrupt: 